# finalg small groups: all groups of order 1 through 40

A folder of **181 finite groups, one for every isomorphism class of order 1 to 40**, serialized by finalg as pretty-printed JSON, together with Jupyter notebooks that describe each group and show the finalg code that builds it.

| What | Where |
|---|---|
| the groups, one JSON file each (`G08_04_Q8.json`) | `json/` |
| the catalogue of all 181 groups | `index.json` |
| loader, pretty-JSON writer, construction helpers | `small_groups_tools.py` |
| one entry per group, with finalg code | `01_orders_01-12.ipynb`, `02_orders_13-20.ipynb`, `03_orders_21-31.ipynb`, `04_order_32.ipynb`, `05_orders_33-40.ipynb` |
| GAP routine that writes a GAP group as finalg JSON | `gap_to_finalg.g` |

**Names and ids.** Every group has a *SmallGroup id* `(order, index)` that follows GAP's SmallGroup library, e.g. `(8, 4)` is the quaternion group, and a *finalg name* such as `Q8`. Orders 1 to 20 use exactly the names of `finalg.small_groups()`. Both can be used to load a group.

**How the groups were checked.** Every group was built from the finalg code in the notebooks (except three, below), its Cayley table was exported to GAP, and GAP's `IdGroup` had to return the id of the group's file name. All 181 agree, so each isomorphism class occurs exactly once, and the number of groups of each order is the known one (OEIS A000001).

**Requirements.** `finalg` (with `numpy` and its other dependencies). `hyprat` is needed only for the optional section below, and GAP only if you want to convert more groups.

## 1. The census

In [1]:
import sys
sys.path.insert(0, '.')            # so that small_groups_tools can be imported from this folder

import small_groups_tools as sgt

index = sgt.groups_index()
print(len(index), "groups\n")
print("order  groups  abelian  from finalg code  from GAP")
for n in sorted({r['order'] for r in index}):
    rs = [r for r in index if r['order'] == n]
    print(f"{n:5d}  {len(rs):6d}  {sum(r['abelian'] for r in rs):7d}  "
          f"{sum(r['source'] == 'finalg' for r in rs):16d}  {sum(r['source'] == 'gap' for r in rs):8d}")

Finite Algebras:
  To see a listing of built-in example algebras, run finalg.examples.about()
  To retrieve an example algebra, use finalg.examples[INDEX]

181 groups

order  groups  abelian  from finalg code  from GAP
    1       1        1                 1         0
    2       1        1                 1         0
    3       1        1                 1         0
    4       2        2                 2         0
    5       1        1                 1         0
    6       2        1                 2         0
    7       1        1                 1         0
    8       5        3                 5         0
    9       2        2                 2         0
   10       2        1                 2         0
   11       1        1                 1         0
   12       5        2                 5         0
   13       1        1                 1         0
   14       2        1                 2         0
   15       1        1                 1         0
   16      14   

Order 32 has 51 groups, more than all of the other orders from 21 to 40 put together (76 groups), which is why it has a notebook of its own.

## 2. Loading the groups

`small_groups_tools` has three loading functions. A group can be named in any of these ways: by name (`'Q8'`), by SmallGroup id (`(8, 4)`), by file stem (`'G08_04'`), or by the path of its JSON file. The loader uses `finalg.make_finite_algebra`, so what comes back is an ordinary finalg `Group`.

In [2]:
# One group
q8 = sgt.load_group('Q8')
print(q8.name, q8.order, type(q8).__name__)
print(sgt.load_group((8, 4)) is not sgt.load_group((8, 4)), "<- each call reads the file again and returns a new object")

Q8 8 Group
True <- each call reads the file again and returns a new object


In [3]:
# A list of groups, in the order of (order, SmallGroup index)
print([g.name for g in sgt.load_groups(orders=8)])
print([g.name for g in sgt.load_groups(names=['Pauli', 'F21', 'Q8'])])
print([g.name for g in sgt.load_groups(keys=[(32, 8), 'D4', 'G24_03'])])

# A dictionary whose keys are the SmallGroup ids
by_id = sgt.load_groups(orders=range(1, 21), as_dict=True)
print(len(by_id), "groups of order 1-20;", by_id[(16, 13)].name, "is (16, 13)")

# Everything: 181 groups
everything = sgt.load_groups()
print(len(everything), "groups loaded")

['Z8', 'Z4_x_Z2', 'D4', 'Q8', 'Z2_x_Z2_x_Z2']
['Q8', 'Pauli', 'F21']
['SG32_8', 'D4', 'SL2_3']
54 groups of order 1-20; Pauli is (16, 13)
181 groups loaded


In [4]:
# Sanity check of the whole library against index.json
sgt.verify_library()

OK: 181 groups of orders 1 through 40.


181

Because every group is loaded from a Cayley table, a loaded group is a first-class finalg object and works with everything else in the library, e.g. `about_subalgebras`:

In [5]:
from finalg.about_subalgebras import about_subalgebras

_ = about_subalgebras(sgt.load_group('Q8'))


Subalgebras of Q8 : Autogenerated dicyclic group Dic2 of order 8 (the quaternion group)

  There are 2 unique proper subalgebras, up to isomorphism, out of 4 total subalgebras.
  as shown below:

1 Commutative Normal Group of order 2 with identity 'e':
   (isomorphic to the small group Z2)
      Group: Q8_subalgebra_3: ('e', 'a^2')

3 Isomorphic Commutative Normal Groups of order 4 with identity 'e':
   (each isomorphic to the small group Z4)
      Group: Q8_subalgebra_0: ('e', 'a^2', 'x', 'a^2x')
      Group: Q8_subalgebra_1: ('e', 'a^2', 'ax', 'a^3x')
      Group: Q8_subalgebra_2: ('e', 'a', 'a^2', 'a^3')



## 3. The JSON format

The files are what `FiniteAlgebra.to_dict()` returns (`name`, `description`, `elements`, `table`), laid out for reading: one key per line, the element names wrapped to 100 columns, and **one row of the Cayley table per line**. `make_finite_algebra(path)` reads them back (the extra whitespace is ordinary JSON whitespace). `small_groups_tools.dump_pretty(alg, path)` writes any finalg algebra in this layout.

(`FiniteAlgebra.dump` currently writes a single line, which is hard to read for anything but the smallest groups. `dump_pretty` could replace it.)

In [6]:
print(open('json/G08_04_Q8.json').read())

{
  "name": "Q8",
  "description": "Autogenerated dicyclic group Dic2 of order 8 (the quaternion group)",
  "elements": ["e", "a", "a^2", "a^3", "x", "ax", "a^2x", "a^3x"],
  "table": [
    [0, 1, 2, 3, 4, 5, 6, 7],
    [1, 2, 3, 0, 5, 6, 7, 4],
    [2, 3, 0, 1, 6, 7, 4, 5],
    [3, 0, 1, 2, 7, 4, 5, 6],
    [4, 7, 6, 5, 2, 1, 0, 3],
    [5, 4, 7, 6, 3, 2, 1, 0],
    [6, 5, 4, 7, 0, 3, 2, 1],
    [7, 6, 5, 4, 1, 0, 3, 2]
  ]
}



In [7]:
# Write any finalg algebra in the same layout, and read it back
import tempfile, os
from finalg import make_finite_algebra

g = sgt.load_group('D4') * sgt.load_group('Z3')
path = os.path.join(tempfile.mkdtemp(), 'D4_x_Z3.json')
sgt.dump_pretty(g, path)
h = make_finite_algebra(path)
print(h.name, h.order, h.to_dict() == g.to_dict())

D4_x_Z3 24 True


## 4. How the groups are built

Orders 1 to 20 are exactly the catalogue in `finalg/small_algebras.py`. For orders 21 to 40 (127 groups), GAP was used to find, for each group, a construction in terms of what finalg already has. The notebooks show the code for every one of them:

| Kind | How | Groups of order 21-40 |
|---|---|---|
| cyclic | `generate_cyclic_group(n)` | 20 |
| direct product (abelian or not) | `G1 * G2`, and `z(n) ** k`, of smaller groups | 47 |
| dihedral | `generate_dihedral_group` | 10 |
| dicyclic (incl. Q16, Q32) | `generate_dicyclic_group` | 5 |
| symmetric | `generate_symmetric_group(4)` (S4) | 1 |
| semi-direct product | `Group.semidirect_product`, with `mult_action(m, k)` when both factors are cyclic and otherwise `action_from_generators` | 41 |
| **none of these** | converted from GAP | **3** |

For a group that is a semi-direct product of **cyclic** groups, the code states the action as `x -> k x`. In the other cases the code uses `action_from_generators(N, H, images)`: `images` says where the generators of `H` send generators of `N`, and the function extends this to the whole action. `Group.semidirect_product` then checks that it is a homomorphism into the automorphisms of `N`, so a wrong action cannot silently produce a wrong group.

In [8]:
# For example, SL(2,3) = Q8 X| Z3 (the binary tetrahedral group)
from finalg import generate_cyclic_group as z
from small_groups_tools import action_from_generators

N = sgt.load_group('Q8')
H = z(3)
action = action_from_generators(N, H, {'1': {'a': 'ax', 'x': 'a'}})   # the order-3 automorphism a -> ax -> ..., x -> a
G = N.semidirect_product(H, action, name='SL2_3')
print(G.name, G.order, G.fast_isomorphic(sgt.load_group('SL2_3')) is not False)

SL2_3 24 True


## 5. The three groups that have no finalg construction

Three groups of order 32 are neither a direct product, a semi-direct product, a central product, nor a dihedral or dicyclic group: no proper normal subgroup of them has a complement, and they are not generated by two commuting proper subgroups. They are in the library as `SG32_8`, `SG32_15` and `SG32_32`, and are the only files that were not made by finalg code.

Q32 (`SmallGroup(32, 20)`) belongs to the same family of "non-split" groups, but finalg already has it: it is the dicyclic group `generate_dicyclic_group(8)`.

All numbers below were computed with GAP.

| | `SG32_8` | `SG32_15` | `SG32_32` |
|---|---|---|---|
| GAP's description | C2 . ((C4 x C2) : C2) = (C2 x C2) . (C4 x C2) | C4 . D8 = C4 . (C4 x C2) | (C2 x C2) . (C2 x C2 x C2) |
| centre | C2 | C4 | C2 x C2 |
| derived subgroup G' | C2 x C2 | C4 | C2 x C2 |
| abelianization G/G' | C2 x C4 | C2 x C4 | C2 x C2 x C2 |
| exponent | 8 | 8 | 4 |
| nilpotency class | 3 | 3 | 2 |
| minimal number of generators | 2 | 2 | 3 |
| conjugacy classes | 11 | 14 | 14 |
| elements of order 2 / 4 / 8 | 3 / 12 / 16 | 3 / 4 / 24 | 3 / 28 / 0 |
| maximal subgroups | SG(16,6) x 2, SG(16,12) | SG(16,5), SG(16,6) x 2 | SG(16,2), SG(16,4) x 6 |

(`G . H` means a non-split extension of G by H: G is a normal subgroup with quotient H, but H does not embed in the group as a complement.)

**Presentations.** GAP's polycyclic presentations, in the generators `g1, ..., g5` (each relation reads "word = 1"; they are also what the element names of the JSON files are made of):

* `SG32_8`: g1² = g4, g2² = g5, g3² = 1, g4² = g5, g5² = 1, [g1, g2] = g3, [g1, g3] = g5, [g2, g4] = g5, and all other pairs commute.
* `SG32_15`: g1² = g4, g2² = g3, g3² = g5, g4² = g5, g5² = 1, [g1, g2] = g3, [g1, g3] = g5, and all other pairs commute.
* `SG32_32`: g1² = g4, g2² = g5, g3² = g4, g4² = 1, g5² = 1, [g1, g2] = g4, [g1, g3] = g5, and all other pairs commute.

Here `[a, b] = a⁻¹b⁻¹ab`.

### Can they be taken from GAP and converted to finalg's format? Yes, and that is what was done.

`gap_to_finalg.g` defines `FinalgJSON(G, name, description, filename)`, which writes any GAP group as a pretty-printed finalg JSON file. Elements of a solvable group are named by their polycyclic exponent vectors (`g1^2g4`; the identity is `e`), and the identity is the first element, as finalg expects. In GAP:

```gap
gap> Read("gap_to_finalg.g");
gap> FinalgJSON(SmallGroup(32, 8), "SG32_8", "GAP SmallGroup(32,8)", "G32_08_SG32_8.json");
```

and then in Python, `make_finite_algebra("G32_08_SG32_8.json")`, or `load_group('SG32_8')`. The routine works for any group GAP can list the elements of, so the same step covers the groups of order 41 and above that finalg cannot construct. GAP is not needed to *use* the library, only to add to it.

In [9]:
for name in ('SG32_8', 'SG32_15', 'SG32_32'):
    g = sgt.load_group(name)
    print(name, g.order, "abelian:", g.is_commutative(), "| elements:", list(g.elements)[:5], "...")
print()
print(sgt.load_group('SG32_8').description)

SG32_8 32 abelian: False | elements: ['e', 'g5', 'g4', 'g4g5', 'g3'] ...
SG32_15 32 abelian: False | elements: ['e', 'g5', 'g4', 'g4g5', 'g3'] ...
SG32_32 32 abelian: False | elements: ['e', 'g5', 'g4', 'g4g5', 'g3'] ...

GAP SmallGroup(32,8) = C2 . ((C4 x C2) : C2) = (C2 x C2) . (C4 x C2). No finalg construction (not a semidirect or central product); converted from GAP with gap_to_finalg.g. Elements are polycyclic exponent vectors g1^a1...g5^a5.


A route that was *not* taken: finalg could also build these groups from permutations (GAP can give a permutation representation of each) with `generate_algebra_from_element_dict` and finalg's `Perm`, which would make them look like the other generator-based groups. That needs a faithful permutation representation of each group and a decision about element names, so the Cayley-table conversion above was used.

## 6. Groups of units with `hyprat` (optional)

[`hyprat`](https://github.com/alreich/hyper_rationals) adds one dependency (`pip install git+https://github.com/alreich/hyper_rationals.git`) and gives exact hypercomplex numbers. Its `units()` methods return a dictionary of the unit elements, and that is exactly the input that `generate_algebra_from_element_dict` expects, so groups of units become Cayley tables in two lines:

* `Hy.units(2)`: the 8 units ±1, ±i, ±j, ±k of the rational quaternions, which form the quaternion group Q8.
* `Hu.units()`: the 24 units of the Hurwitz integers (±1, ±i, ±j, ±k and (±1 ±i ±j ±k)/2), which form the binary tetrahedral group 2T = SL(2,3).

(`closed=True` says that the dictionary already contains the whole group, so that finalg does not need to close it under multiplication.)

In [10]:
try:
    from hyprat import Hy, Hu
except ImportError:
    print("hyprat is not installed: pip install git+https://github.com/alreich/hyper_rationals.git")
else:
    from finalg import generate_algebra_from_element_dict

    Q8h, *_ = generate_algebra_from_element_dict(
        Hy.units(2), name='Q8_hyprat', description='The 8 unit quaternions', closed=True)
    T24h, *_ = generate_algebra_from_element_dict(
        Hu.units(), name='BinaryTetrahedral', description='The 24 unit Hurwitz quaternions', closed=True)

    print(Q8h.name, Q8h.order, type(Q8h).__name__, "| elements:", list(Q8h.elements))
    print(T24h.name, T24h.order, type(T24h).__name__)

    # They are the groups in the library
    print("Q8 from units is Q8:", Q8h.fast_isomorphic(sgt.load_group('Q8')) is not False)
    print("2T from units is SL2_3 (24, 3):", T24h.fast_isomorphic(sgt.load_group((24, 3))) is not False)

Q8_hyprat 8 Group | elements: ['1', '-1', 'i', '-i', 'j', '-j', 'k', '-k']
BinaryTetrahedral 24 Group
Q8 from units is Q8: True


2T from units is SL2_3 (24, 3): True


The element names are the same strings that `hyprat` prints (`'-1'`, `'i'`, `'1/2+1/2i+1/2j+1/2k'`, ...), so each element of the group can be mapped straight back to a quaternion:

In [11]:
try:
    units = Hu.units()
    print(list(T24h.elements)[:10])
    a, b = '1/2+1/2i+1/2j+1/2k', 'j'
    print(a, '*', b, '=', T24h.op(a, b))
    print("same as the quaternion product:", str(units[a] * units[b]).strip('()') , "(hyprat)")
except NameError:
    pass

['1', '-1', 'i', '-i', 'j', '-j', 'k', '-k', '1/2+1/2i+1/2j+1/2k', '1/2+1/2i+1/2j-1/2k']
1/2+1/2i+1/2j+1/2k * j = -1/2-1/2i+1/2j+1/2k
same as the quaternion product: -1/2-1/2i+1/2j+1/2k (hyprat)


The same idea gives other groups of units: `Hy.units(1)` (order 4: Z4), `Hy.units(2, signs="split-quaternion")` (order 8: isomorphic to D4), and, using generators only, the closure of a few Hurwitz units (e.g. `i`, `j` and `1/2+1/2i+1/2j+1/2k`) is again the 24-element group:

In [12]:
try:
    gens = {k: v for k, v in Hu.units().items() if k in ('i', 'j', '1/2+1/2i+1/2j+1/2k')}
    two_t, elem_dict, iterations = generate_algebra_from_element_dict(gens, name='2T', description='closure of 3 generators')
    print(two_t.order, "elements after", iterations, "iterations")
except NameError:
    pass

24 elements after 3 iterations


Note: `Hy.units(3)` (the 16 octonion units) is not a group, since the octonions are not associative; `generate_algebra_from_element_dict` returns a `Loop` for it.

## 7. Notes and suggestions

**Using the JSON files inside finalg (`about_subalgebras`).** Right now `finalg/small_algebras.py` builds the catalogue in code, and `about_subalgebras` gets names through `identify_small_group`. With this library there are two options:

1. Keep the code, and use the JSON files as a cross-check (`verify_library` and the GAP ids).
2. Ship the folder as package data (e.g. `src/finalg/data/small_groups/`; `pyproject.toml` already includes `*.json` for `finalg.data.algebras`), let `small_groups(order)` and `identify_small_group` load from it, and extend `MAX_SMALL_ORDER` to 40. Loading all 181 files takes about a tenth of a second, so no code-built catalogue is needed, and the groups of order 21 to 40 become available to `about_subalgebras` without any new constructions in finalg.

The names of the groups of order 21 to 40 are new and can be changed freely (they are only in `index.json` and the file names); the SmallGroup ids and the GAP structure descriptions in `index.json` are what identify a group.

**Isomorphism testing is the bottleneck at order 32.** `identify_small_group` calls `fast_isomorphic`, which calls `Magma._smallest_generating_set`, which calls `generators()`, whose repeated recursive `closure` takes about 40 seconds for some groups of order 32. A greedy generating set found with a breadth-first closure took the same test to 0.02 seconds, and the existing tests in `test_magma.py`, `test_group.py`, and `test_small_algebras.py` passed with it. This was needed, in a patched form, to check these 181 groups; the library itself needs nothing but `finalg`. A cheaper first test also helps: `index.json` could store invariants (element-order histogram, centre order, derived-subgroup order, number of conjugacy classes) so that most groups are identified without any isomorphism test.

**Not yet done.** Orders above 40; a Cayley-table drawing for each group; storing a presentation in each JSON file (finalg's format has no field for one).